# Data Pipeline: Learning Goal Paper Ingestion

This notebook replicates the pipeline for adding new papers based on each learning goal topic.

It does the following:
- Reads learning goals from Postgres (Lakebase)
- Searches OpenAlex per goal topic
- Reconstructs abstract text
- Generates embeddings with SentenceTransformer
- Upserts papers idempotently by openalex_id
- Links papers to collections and reading progress with idempotent inserts


In [0]:
!pip install -r ../requirements.txt

In [0]:
import base64
import json
import os
from contextlib import contextmanager
from typing import Any, Dict, List, Optional

import psycopg2
import requests
from databricks.sdk import WorkspaceClient
from psycopg2.extras import RealDictCursor
from sentence_transformers import SentenceTransformer


In [0]:
# Lakebase-style Postgres connection (same secret pattern as lakebase.py)
_w = WorkspaceClient()
LAKEBASE_SCOPE = os.environ.get("LAKEBASE_SECRET_SCOPE", "database")
LAKEBASE_KEY = os.environ.get("LAKEBASE_SECRET_KEY", "lakebase-url")

def _decode_secret_value(raw_value: str) -> str:
    if not raw_value:
        return raw_value
    try:
        return base64.b64decode(raw_value).decode("utf-8")
    except Exception:
        return raw_value

def lakebase_url() -> str:
    secret = _w.secrets.get_secret(scope=LAKEBASE_SCOPE, key=LAKEBASE_KEY)
    return _decode_secret_value(secret.value)

@contextmanager
def get_connection():
    conn = psycopg2.connect(lakebase_url(), cursor_factory=RealDictCursor)
    try:
        yield conn
    finally:
        conn.close()

def run_query(sql: str, params: Optional[tuple] = None) -> List[Dict[str, Any]]:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()


In [0]:
# OpenAlex + embedding setup
OPENALEX_BASE_URL = os.environ.get("OPENALEXAPI_API_BASE_URL", "https://api.openalex.org").rstrip("/")
OPENALEX_SCOPE = os.environ.get("OPENALEXAPI_SECRET_SCOPE", "openalex")
OPENALEX_KEY = os.environ.get("OPENALEXAPI_SECRET_KEY", "api-key")

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_CACHE_DIR = "/tmp/.cache/huggingface"

def openalex_api_key() -> Optional[str]:
    try:
        secret = _w.secrets.get_secret(scope=OPENALEX_SCOPE, key=OPENALEX_KEY)
        return _decode_secret_value(secret.value).strip()
    except Exception:
        return None

session = requests.Session()
OPENALEX_KEY_VALUE = openalex_api_key()

def openalex_get(endpoint: str, params: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    params = dict(params or {})
    if OPENALEX_KEY_VALUE:
        params["api_key"] = OPENALEX_KEY_VALUE
    resp = session.get(f"{OPENALEX_BASE_URL}/{endpoint.lstrip('/')}", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

def reconstruct_abstract(inverted_index: Optional[Dict[str, List[int]]]) -> str:
    if not inverted_index:
        return ""
    pairs: List[tuple] = []
    for token, positions in inverted_index.items():
        for p in positions:
            pairs.append((p, token))
    pairs.sort(key=lambda x: x[0])
    return " ".join(token for _, token in pairs)

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=MODEL_CACHE_DIR)
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")


In [0]:
# SQL helpers
def fetch_learning_goals(user_id: Optional[str] = None) -> List[Dict[str, Any]]:
    if user_id:
        return run_query(
            """
            SELECT id, user_id, title, description, status
            FROM learning_goals
            WHERE user_id = %s AND status = 'active'
            ORDER BY created_at DESC
            """,
            (user_id,)
        )
    return run_query(
        """
        SELECT id, user_id, title, description, status
        FROM learning_goals
        WHERE status = 'active'
        ORDER BY created_at DESC
        """
    )

def normalize_openalex_id(raw_id: Optional[str]) -> Optional[str]:
    if not raw_id:
        return None
    return raw_id.replace("https://openalex.org/", "")

def vector_literal(vec: Optional[List[float]]) -> Optional[str]:
    if vec is None:
        return None
    return "[" + ",".join(f"{x:.8f}" for x in vec) + "]"


In [0]:
# Idempotent upsert + linking pipeline
def ensure_goal_collection(cur, goal: Dict[str, Any]) -> str:
    cur.execute(
        """
        SELECT id
        FROM collections
        WHERE user_id = %s AND learning_goal_id = %s
        ORDER BY created_at DESC
        LIMIT 1
        """,
        (goal["user_id"], goal["id"])
    )
    row = cur.fetchone()
    if row:
        return row["id"]

    cur.execute(
        """
        INSERT INTO collections (user_id, learning_goal_id, name, description)
        VALUES (%s, %s, %s, %s)
        RETURNING id
        """,
        (goal["user_id"], goal["id"], goal["title"], goal.get("description"))
    )
    return cur.fetchone()["id"]

def upsert_venue(cur, source: Dict[str, Any]) -> Optional[str]:
    if not source or not source.get("id"):
        return None
    venue_openalex_id = normalize_openalex_id(source.get("id"))
    cur.execute(
        """
        INSERT INTO venues (openalex_id, display_name, issn_l, issn, is_oa, type, host_organization, updated_at)
        VALUES (%s, %s, %s, %s, %s, %s, %s, NOW())
        ON CONFLICT (openalex_id) DO UPDATE
        SET display_name = COALESCE(EXCLUDED.display_name, venues.display_name), updated_at = NOW()
        RETURNING id
        """,
        (
            venue_openalex_id,
            source.get("display_name"),
            source.get("issn_l"),
            json.dumps(source.get("issn", [])),
            source.get("is_oa", False),
            source.get("type"),
            source.get("host_organization_name")
        )
    )
    return cur.fetchone()["id"]

def upsert_paper(cur, work: Dict[str, Any], abstract_embedding: Optional[List[float]]) -> Optional[str]:
    openalex_id = normalize_openalex_id(work.get("id"))
    if not openalex_id:
        return None

    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}
    venue_id = upsert_venue(cur, source)

    abstract_text = reconstruct_abstract(work.get("abstract_inverted_index"))
    emb = vector_literal(abstract_embedding)

    cur.execute(
        """
        INSERT INTO papers (
            openalex_id, doi, title, display_name, abstract, abstract_inverted_index,
            publication_year, publication_date, type, language,
            venue_id, venue_display_name,
            is_oa, oa_status, pdf_url, landing_page_url,
            cited_by_count, keywords, concepts, topics,
            abstract_embedding, abstract_embedding_model, abstract_embedding_generated_at, updated_at
        )
        VALUES (
            %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s, %s,
            %s, %s, %s, %s,
            %s, %s, %s, %s,
            %s::vector, %s, NOW(), NOW()
        )
        ON CONFLICT (openalex_id) DO UPDATE
        SET
            title = COALESCE(EXCLUDED.title, papers.title),
            display_name = COALESCE(EXCLUDED.display_name, papers.display_name),
            abstract = COALESCE(EXCLUDED.abstract, papers.abstract),
            cited_by_count = COALESCE(EXCLUDED.cited_by_count, papers.cited_by_count),
            abstract_embedding = COALESCE(EXCLUDED.abstract_embedding, papers.abstract_embedding),
            abstract_embedding_model = COALESCE(EXCLUDED.abstract_embedding_model, papers.abstract_embedding_model),
            abstract_embedding_generated_at = CASE WHEN EXCLUDED.abstract_embedding IS NOT NULL THEN NOW() ELSE papers.abstract_embedding_generated_at END,
            updated_at = NOW()
        RETURNING id
        """,
        (
            openalex_id,
            work.get("doi"),
            work.get("title") or work.get("display_name"),
            work.get("display_name"),
            abstract_text or None,
            json.dumps(work.get("abstract_inverted_index") or {}),
            work.get("publication_year"),
            work.get("publication_date"),
            work.get("type"),
            work.get("language"),
            venue_id,
            source.get("display_name"),
            (work.get("open_access") or {}).get("is_oa", False),
            (work.get("open_access") or {}).get("oa_status"),
            primary_location.get("pdf_url"),
            primary_location.get("landing_page_url"),
            work.get("cited_by_count", 0),
            json.dumps(work.get("keywords") or []),
            json.dumps(work.get("concepts") or []),
            json.dumps(work.get("topics") or []),
            emb,
            EMBEDDING_MODEL_NAME if emb else None
        )
    )
    return cur.fetchone()["id"]

def link_paper_to_goal(cur, user_id: str, goal_id: str, collection_id: str, paper_id: str):
    cur.execute(
        """
        INSERT INTO collection_papers (collection_id, paper_id)
        VALUES (%s, %s)
        ON CONFLICT (collection_id, paper_id) DO NOTHING
        """,
        (collection_id, paper_id)
    )

    cur.execute(
        """
        INSERT INTO reading_progress (user_id, paper_id, learning_goal_id, status)
        VALUES (%s, %s, %s, 'not_started')
        ON CONFLICT (user_id, paper_id) DO UPDATE
        SET learning_goal_id = EXCLUDED.learning_goal_id, updated_at = NOW()
        """,
        (user_id, paper_id, goal_id)
    )


In [0]:
def search_openalex_for_goal(goal: Dict[str, Any], per_page: int = 25) -> List[Dict[str, Any]]:
    query = (goal.get("title") or "").strip()
    desc = (goal.get("description") or "").strip()
    if desc:
        query = f"{query} {desc}"
    payload = openalex_get(
        "works",
        params={"search": query, "per_page": per_page, "sort": "cited_by_count:desc"}
    )
    return payload.get("results", []) if payload else []

def run_goal_paper_pipeline(user_id: Optional[str] = None, per_goal_limit: int = 25, embed_new_only: bool = True, dry_run: bool = False) -> Dict[str, Any]:
    goals = fetch_learning_goals(user_id=user_id)
    stats = {"goals_processed": 0, "works_seen": 0, "papers_upserted": 0, "links_created_or_refreshed": 0, "errors": 0}

    if not goals:
        print("No active learning goals found.")
        return stats

    with get_connection() as conn:
        with conn.cursor() as cur:
            for goal in goals:
                stats["goals_processed"] += 1
                print(f"\n=== Goal: {goal['title']} ({goal['id']}) ===")

                try:
                    works = search_openalex_for_goal(goal, per_page=per_goal_limit)
                except Exception as e:
                    stats["errors"] += 1
                    print(f"OpenAlex error for goal {goal['id']}: {e}")
                    continue

                collection_id = None if dry_run else ensure_goal_collection(cur, goal)

                for work in works:
                    stats["works_seen"] += 1
                    openalex_id = normalize_openalex_id(work.get("id"))
                    if not openalex_id:
                        continue

                    try:
                        abstract_text = reconstruct_abstract(work.get("abstract_inverted_index"))
                        abstract_embedding = None
                        if abstract_text and (not embed_new_only or openalex_id):
                            abstract_embedding = embedder.encode(abstract_text, show_progress_bar=False, normalize_embeddings=True).tolist()

                        if dry_run:
                            continue

                        paper_id = upsert_paper(cur, work, abstract_embedding)
                        if not paper_id:
                            continue

                        stats["papers_upserted"] += 1
                        link_paper_to_goal(cur, goal["user_id"], goal["id"], collection_id, paper_id)
                        stats["links_created_or_refreshed"] += 1

                    except Exception as e:
                        stats["errors"] += 1
                        print(f"Error processing work {openalex_id}: {e}")

                if not dry_run:
                    conn.commit()

    print("\nPipeline completed.")
    print(json.dumps(stats, indent=2))
    return stats


In [0]:
# Main daily routine (entrypoint for a scheduled job)
from datetime import datetime, timezone

def _env_bool(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

def main_daily_pipeline() -> Dict[str, Any]:
    run_ts = datetime.now(timezone.utc).isoformat()
    user_id = os.environ.get("PIPELINE_USER_ID")  # Optional: run for one user only
    per_goal_limit = int(os.environ.get("PIPELINE_PER_GOAL_LIMIT", "25"))
    embed_new_only = _env_bool("PIPELINE_EMBED_NEW_ONLY", True)
    dry_run = _env_bool("PIPELINE_DRY_RUN", False)

    print(f"Starting daily pipeline run at {run_ts}")
    print(
        f"Config: user_id={user_id or 'ALL'}, per_goal_limit={per_goal_limit}, "
        f"embed_new_only={embed_new_only}, dry_run={dry_run}"
    )

    stats = run_goal_paper_pipeline(
        user_id=user_id,
        per_goal_limit=per_goal_limit,
        embed_new_only=embed_new_only,
        dry_run=dry_run
    )

    # Fail the scheduled job if errors were observed.
    if stats.get("errors", 0) > 0:
        raise RuntimeError(f"Daily pipeline completed with errors: {stats}")

    print("Daily pipeline completed successfully.")
    return stats

# Execute when this notebook cell runs (use this cell as the scheduled task entrypoint).
DAILY_PIPELINE_STATS = main_daily_pipeline()